# 01 — Load & Inspect Raw Data
Load the two raw CSVs, do a basic schema/quality inspection, then write cleaned versions to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np

## 1. Load raw files

In [2]:
stock_prices = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\raw\stock_prices.csv")
financials = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\raw\financials.csv")

print("stock_prices:", stock_prices.shape)
print("financials  :", financials.shape)

stock_prices: (2332531, 12)
financials  : (92956, 45)


C:\Users\naksh\AppData\Local\Temp\ipykernel_43056\2463330433.py:2: DtypeWarning: Columns (0: OrdinaryProfit, 1: Profit, 2: EarningsPerShare, 3: TotalAssets, 4: Equity, 5: EquityToAssetRatio, 6: NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock, 7: AverageNumberOfShares) have mixed types. Specify dtype option on import or set low_memory=False.
  financials = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\raw\financials.csv")


## 2. Schema inspection

In [3]:
print("=== stock_prices columns ===")
print(stock_prices.columns.tolist())

print("\n=== financials columns ===")
print(financials.columns.tolist())

=== stock_prices columns ===
['RowId', 'Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close', 'Volume', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag', 'Target']

=== financials columns ===
['DisclosureNumber', 'DateCode', 'Date', 'SecuritiesCode', 'DisclosedDate', 'DisclosedTime', 'DisclosedUnixTime', 'TypeOfDocument', 'CurrentPeriodEndDate', 'TypeOfCurrentPeriod', 'CurrentFiscalYearStartDate', 'CurrentFiscalYearEndDate', 'NetSales', 'OperatingProfit', 'OrdinaryProfit', 'Profit', 'EarningsPerShare', 'TotalAssets', 'Equity', 'EquityToAssetRatio', 'BookValuePerShare', 'ResultDividendPerShare1stQuarter', 'ResultDividendPerShare2ndQuarter', 'ResultDividendPerShare3rdQuarter', 'ResultDividendPerShareFiscalYearEnd', 'ResultDividendPerShareAnnual', 'ForecastDividendPerShare1stQuarter', 'ForecastDividendPerShare2ndQuarter', 'ForecastDividendPerShare3rdQuarter', 'ForecastDividendPerShareFiscalYearEnd', 'ForecastDividendPerShareAnnual', 'ForecastNetSales', 'ForecastOperatingProfit

In [4]:
stock_prices.info()
financials.info()

<class 'pandas.DataFrame'>
RangeIndex: 2332531 entries, 0 to 2332530
Data columns (total 12 columns):
 #   Column            Dtype  
---  ------            -----  
 0   RowId             str    
 1   Date              str    
 2   SecuritiesCode    int64  
 3   Open              float64
 4   High              float64
 5   Low               float64
 6   Close             float64
 7   Volume            int64  
 8   AdjustmentFactor  float64
 9   ExpectedDividend  float64
 10  SupervisionFlag   bool   
 11  Target            float64
dtypes: bool(1), float64(7), int64(2), str(2)
memory usage: 198.0 MB
<class 'pandas.DataFrame'>
RangeIndex: 92956 entries, 0 to 92955
Data columns (total 45 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------                                                                        --------------  -----  
 0   DisclosureNumber                                                              

## 3. Missing-value rates

In [5]:
print("=== stock_prices — missing rate ===")
print(stock_prices.isna().mean().sort_values(ascending=False))

print("\n=== financials — missing rate ===")
print(financials.isna().mean().sort_values(ascending=False))

=== stock_prices — missing rate ===
ExpectedDividend    0.991912
Open                0.003262
High                0.003262
Close               0.003262
Low                 0.003262
Target              0.000102
RowId               0.000000
Date                0.000000
SecuritiesCode      0.000000
Volume              0.000000
AdjustmentFactor    0.000000
SupervisionFlag     0.000000
dtype: float64

=== financials — missing rate ===
ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements                   0.922017
ForecastDividendPerShare1stQuarter                                              0.793010
ResultDividendPerShareAnnual                                                    0.791138
ResultDividendPerShareFiscalYearEnd                                             0.791127
BookValuePerShare                                                               0.615162
ResultDividendPerShare3rdQuarter                                                0.594679
ForecastDividendPerShare2ndQuart

## 4. Parse dates

In [6]:
stock_prices["Date"] = pd.to_datetime(stock_prices["Date"])

for col in ["Date", "DisclosedDate"]:
    if col in financials.columns:
        financials[col] = pd.to_datetime(financials[col])

## 5. Duplicate & universe checks

In [7]:
print("stock_prices duplicates:", stock_prices.duplicated().sum())
print("financials   duplicates:", financials.duplicated().sum())

price_codes    = set(stock_prices["SecuritiesCode"].unique())
financial_codes = set(financials["SecuritiesCode"].unique())
common_codes   = price_codes & financial_codes

print(f"\nTickers in prices       : {len(price_codes)}")
print(f"Tickers in financials   : {len(financial_codes)}")
print(f"Tickers in both         : {len(common_codes)}")
print(f"Only in financials      : {len(financial_codes - price_codes)}")

stock_prices duplicates: 0
financials   duplicates: 0

Tickers in prices       : 2000
Tickers in financials   : 4072
Tickers in both         : 2000
Only in financials      : 2072


## 6. Save processed files

In [8]:
stock_prices.to_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processedstock_prices_clean.csv", index=False)
financials.to_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processedfinancials_clean.csv",   index=False)

print("Saved: stock_prices_clean.csv")
print("Saved: financials_clean.csv")

Saved: stock_prices_clean.csv
Saved: financials_clean.csv
